# Exercise: Training a Neural Forecasting Model

A streaming service needs to forecast monthly subscribers. Growth follows an S-curve that ARIMA struggles with. Train an N-BEATS model with quantile regression and compare it to an ARIMA baseline.

In [ ]:
import os
import warnings
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Udacity brand colors
UB = {
    "Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"
}
UN = {
    "Black": "#0A0B0F", "Dark Gray": "#5A5C62",
    "Medium Gray": "#9C9FA8", "Gray": "#CECFD4",
    "Light Gray": "#F2F2F2", "White": "#FFFFFF"
}
US = {
    "Orange": "#EE7622", "Yellow": "#F9DC5C",
    "Red": "#9C0D22", "Green": "#23CE6B"
}

mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import NBEATSModel, ARIMA as DartsARIMA
from darts.metrics import mae, rmse, mape
from darts.utils.likelihood_models import QuantileRegression

HORIZON = 12

In [ ]:
def fit_nbeats(train, val, epochs=50):
    """
    Fit an N-BEATS model on the training series.
    
    Parameters
    ----------
    train : darts.TimeSeries
    val : darts.TimeSeries
    epochs : int
    
    Returns
    -------
    darts.models.NBEATSModel
        Fitted model.
    """
    model = NBEATSModel(
        input_chunk_length=...,
        output_chunk_length=...,
        generic_architecture=True,
        num_blocks=...,
        num_layers=...,
        layer_widths=...,
        n_epochs=epochs,
        random_state=42
    )
    model.fit(train, val_series=val)
    return model

nbeats_model = fit_nbeats(train, val)

In [ ]:
def plot_nbeats_forecast(train, val, forecast):
    """
    Plot N-BEATS forecast against actual validation values.
    
    Parameters
    ----------
    train, val, forecast : darts.TimeSeries
    """
    fig, ax = plt.subplots(figsize=(14, 6))
    train.plot(ax=ax, label="Train", color=UN["Black"])
    val.plot(ax=ax, label="Actual", color=UB["Brand Blue"])
    forecast.plot(ax=ax, label="N-BEATS", color=US["Orange"])
    ax.set_title("N-BEATS Forecast", fontsize=18, fontweight="bold")
    ax.set_ylabel("Demand (MWh)", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.legend()
    plt.tight_layout()
    plt.show()

nbeats_fc = nbeats_model.predict(len(val))
plot_nbeats_forecast(train, val, nbeats_fc)

In [ ]:
arima = DartsARIMA(p=2, d=1, q=1)
arima.fit(train)
arima_fc = arima.predict(HORIZON)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
TimeSeries.from_series(subscribers).plot(ax=ax, label="Actual", color=UN["Black"])
arima_fc.plot(ax=ax, label="ARIMA(2,1,1)", color=UB["Brand Blue"])
nbeats_fc.plot(ax=ax, label="N-BEATS", color=US["Orange"], low_quantile=0.05, high_quantile=0.95)
ax.set_ylabel("Subscribers")
ax.legend(loc="upper left")
plt.tight_layout()
ax.set_title("Plot", fontsize=18, fontweight="bold")
ax.tick_params(axis="both", labelsize=14)
plt.show()

In [ ]:
print("=== Forecast at Month 12 ===")
print(f"  ARIMA: {arima_fc.values().flatten()[-1]:,.0f}")
print(f"  N-BEATS: {nbeats_fc.values().flatten()[-1]:,.0f}")